In [ ]:
'''
State the difference between a Primary key and a Candidate key.

A Candidate Key is any column or combination of columns that can uniquely identify each record within the table. A table can have multiple candidate keys. Each candidate key qualifies to become the primary key.

A Primary Key is the one candidate key that the designer selects as the official row identifier. A table can have only one primary key. The primary key column prohibits null entries and does not allow duplicate values.

A table can have only one primary key but multiple candidate keys. The primary key may not contain null entries, whereas a candidate key that is not chosen as the primary key may allow NULLs depending on the design. All primary keys are candidate keys, but not all candidate keys are primary keys. The primary key is automatically enforced by the RDBMS through a unique index, and it is commonly used as the target of foreign key references from other tables.


Explain any two string functions with an example.

String functions perform operations on character data and return either a string or a numeric result.

CONCAT() joins two or more strings into one. Syntax: CONCAT(string1, string2, ...) Example: SELECT CONCAT('Hello', ' ', 'World'); returns Hello World. In practice: SELECT CONCAT(first_name, ' ', last_name) AS full_name FROM employees; builds a full name from two columns.

SUBSTRING() extracts a part of a string from a given starting position for a given number of characters. Syntax: SUBSTRING(string, start, length). Positions begin at 1. Example: SELECT SUBSTRING('Database', 1, 4); returns Data.

LENGTH() returns the number of characters in a string. Example: SELECT LENGTH('Hello'); returns 5.

UPPER() and LOWER() convert a string to all uppercase or all lowercase. Example: SELECT UPPER('hello'); returns HELLO. SELECT LOWER('WORLD'); returns world.

TRIM() removes leading and trailing whitespace. Example: SELECT TRIM('  hello  '); returns hello.

REPLACE() substitutes every occurrence of a substring with another string. Example: SELECT REPLACE('Hello World', 'World', 'SQL'); returns Hello SQL.


Can we use aggregate functions in where clause? Justify your answer.

No. Aggregate functions such as COUNT(), SUM(), AVG(), MIN(), and MAX() may not be used directly inside the WHERE clause.

The reason is the order of SQL clause evaluation. The WHERE clause runs before GROUP BY and before any aggregation takes place. Aggregate functions compute their results across groups of rows, so their values simply do not exist yet when WHERE is being evaluated.

The following query is invalid: SELECT department_id, COUNT(*) FROM employees WHERE COUNT(*) > 5 GROUP BY department_id; MySQL returns an error because COUNT(*) cannot appear in the WHERE clause.

The correct approach is to use the HAVING clause, which filters groups after GROUP BY and aggregation have been applied: SELECT department_id, COUNT(*) FROM employees GROUP BY department_id HAVING COUNT(*) > 5;

HAVING is specifically designed to filter on aggregate results, while WHERE filters individual rows before any grouping.


What is Cross-Join?

A CROSS JOIN (also called a Cartesian join) combines each individual row from the first table with every row from the second table. If the first table has M rows and the second has N rows, the result has M × N rows. There is no ON condition — every possible pair is produced.

Syntax: SELECT * FROM table1 CROSS JOIN table2;

Example: If an employees table has 10 rows and a departments table has 5 rows, the CROSS JOIN produces 50 rows.

CROSS JOIN is different from other joins because it requires no any matching condition. It is rarely used in production because it can produce enormous result sets. Legitimate uses include generating all combinations of two sets (such as all size-colour combinations for a product catalogue), creating test data, or building a calendar grid.


Aryan, a database administrator, has grouped records of a table with the help of group by clause. He needs to further filter groups of records generated through group by clause. Suggest suitable clause for it and properly explain its usage with the help of an example.

The suitable clause is HAVING.

The HAVING clause filters groups of rows after the GROUP BY clause has created them. It plays the same role for grouped results that WHERE plays for individual rows. WHERE may not be used here because aggregate functions are evaluated after grouping, and HAVING is evaluated after GROUP BY.

Syntax: SELECT column, aggregate_function(column) FROM table GROUP BY column HAVING condition;

Example: Find all departments where the total salary bill exceeds 50000. SELECT department_id, SUM(salary) AS total_salary FROM employees GROUP BY department_id HAVING SUM(salary) > 50000;

In this example, GROUP BY groups rows by department, SUM calculates the total salary per department, and HAVING retains only those departments where the total exceeds 50000. Without HAVING, you would get every department. HAVING lets you filter on the aggregated result.

The mandatory order of clauses is: WHERE filters rows first, then GROUP BY creates groups, then HAVING filters groups, then ORDER BY sorts the final output.


What are the uses of Window Function?

Window functions (also called analytical functions) carry out computations across a set of related rows — the window — while keeping each record individually visible in the output. Unlike GROUP BY, they do not collapse rows into one summary row.

Ranking rows within groups: RANK(), DENSE_RANK(), and ROW_NUMBER() assign a rank to each record within a partition. As an illustration, ranking employees by salary within each department without collapsing the rows.

Running totals and cumulative aggregates: SUM() OVER or COUNT() OVER with ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW computes a cumulative running total as rows are added.

Accessing previous or next rows: LAG() retrieves the value from a prior row and LEAD() retrieves the value from a following row. This serves to compare this month's sales to last month without a self-join.

Moving averages and rolling calculations: AVG() OVER with a frame clause such as ROWS BETWEEN 2 PRECEDING AND CURRENT ROW produces a sliding window average over a configurable number of rows.

Distribution analysis: PERCENT_RANK() and CUME_DIST() calculate what percentile a row falls in. NTILE() divides the result set into equal-sized buckets.

Finding boundary values within a group: FIRST_VALUE() and LAST_VALUE() return the first or last value within the ordered window partition.

The primary advantage is that all of these calculations return a result for each individual row alongside the original row data, something that GROUP BY aggregation cannot do.


Explain about EXISTS operator.

The EXISTS operator tests whether a subquery returns at least one row. It evaluates to TRUE if the inner query produces one or more rows and FALSE if it produces no rows. EXISTS does not return the actual column values from the subquery — it is purely a membership check.

Syntax: SELECT columns FROM table WHERE EXISTS (subquery);

Example: Find all customers who have placed at least one order. SELECT customer_id, customer_name FROM customers WHERE EXISTS (SELECT 1 FROM orders WHERE orders.customer_id = customers.customer_id);

The inner query SELECT 1 is a convention — EXISTS only cares whether any row is returned, not what the value is. You can write SELECT *, SELECT column_name, or SELECT 1; the result is the same.

NOT EXISTS is the negation. It evaluates to TRUE when the inner query gives back no rows, allowing you to find records that have no match in another table. Example: Find customers who have never placed an order. SELECT customer_id FROM customers WHERE NOT EXISTS (SELECT 1 FROM orders WHERE orders.customer_id = customers.customer_id);

EXISTS is often faster than IN for large datasets because it stops searching as soon as the first matching row is found (short-circuit evaluation), whereas IN must evaluate the entire subquery result.


What is percent rank? Explain with an example.

PERCENT_RANK() is a window function that calculates the relative rank of a row as a value between 0.0 and 1.0, where 0 represents the lowest rank and 1 represents the highest.

Formula: PERCENT_RANK = (rank of the row minus 1) divided by (total number of rows in the partition minus 1).

The first row in any ordering always receives a PERCENT_RANK of 0. The last row always receives 1.0.

Syntax: SELECT column, PERCENT_RANK() OVER (PARTITION BY group_col ORDER BY value_col) AS pct_rank FROM table;

Example: Rank each employee's salary within their department. SELECT employee_id, department_id, salary, PERCENT_RANK() OVER (PARTITION BY department_id ORDER BY salary) AS pct_rank FROM employees;

If a department has five employees with salaries 20000, 30000, 40000, 50000, 60000, their PERCENT_RANK values are 0.0, 0.25, 0.5, 0.75, and 1.0 respectively.

A PERCENT_RANK of 0.75 means that employee's salary is higher than 75 percent of employees in that department.

Common use: filtering the top 10 percent of earners by adding WHERE pct_rank >= 0.9, or identifying which percentile a given value falls in for reporting and analysis.


State and explain in brief types of Locks in database.

Locks are database mechanisms that control concurrent access to data. When multiple transactions attempt to read or modify the same data at the same time, locks prevent conflicts and maintain data consistency.

Locking levels: Row-level lock locks one row, allowing other transactions to access all other rows in the same table at the same time. This is the most granular lock and causes the least contention. Page-level lock locks a block of rows stored together on a data page. Table-level lock locks the entire table, blocking other transactions from accessing any row in it. Simple to manage but significantly reduces concurrency.

Types of locks:

Shared Lock (S Lock, also called Read Lock): A shared lock is acquired when a transaction wants to read data. Multiple transactions can hold shared locks on the same data at the same time because reading alone does not change the data. A shared lock prevents other transactions from acquiring an exclusive (write) lock while reading is in progress.

Exclusive Lock (X Lock, also called Write Lock): An exclusive lock is acquired when a transaction wants to modify data through INSERT, UPDATE, or DELETE. Only one transaction at a time can hold an exclusive lock on a piece of data. No other transaction can read or write the data while an exclusive lock is held, ensuring that only one transaction makes changes at any moment.

Together, shared and exclusive locks implement a concurrency protocol: readers can co-exist (shared locks are compatible), but a writer gets sole access (exclusive locks are not compatible with any other lock).


Explain the different ways you can restrict the data in a column. Demonstrate with Examples.

You restrict data in a column by defining SQL constraints. Constraints are rules enforced at the database level that ensure only valid data is accepted.

NOT NULL prevents a column from holding an empty (NULL) value. Every row must supply a value for the column. Example: CREATE TABLE employees (id INT NOT NULL, name VARCHAR(50) NOT NULL);

UNIQUE ensures every value in the column is distinct across all rows. No two rows can share an the same value. A UNIQUE column can still hold null entries. Example: CREATE TABLE users (email VARCHAR(100) UNIQUE);

PRIMARY KEY combines NOT NULL and UNIQUE. It is the main row identifier. A table may have only one primary key. Example: CREATE TABLE students (student_id INT PRIMARY KEY, name VARCHAR(50));

FOREIGN KEY links a column to the primary key of another table, enforcing referential integrity. Values in the foreign key column must exist in the parent table or be NULL. Example: CREATE TABLE orders ( order_id INT PRIMARY KEY, customer_id INT, FOREIGN KEY (customer_id) REFERENCES customers(customer_id) );

CHECK validates column values against a Boolean condition. Any INSERT or UPDATE violating the condition is rejected. Example: CREATE TABLE products ( price DECIMAL CHECK (price > 0), quantity INT CHECK (quantity >= 0) );

DEFAULT sets an automatic value when no value is explicitly provided during INSERT. Example: CREATE TABLE employees (status VARCHAR(10) DEFAULT 'Active');


What is the difference between primary key and unique key?

A PRIMARY KEY uniquely identifies each individual row within the table. It automatically enforces both the NOT NULL and UNIQUE constraints. A table can have exactly one primary key. The primary key is the main reference point for foreign keys in other tables. The column designated as the primary key may not contain a missing entry under any circumstances.

A UNIQUE KEY (or UNIQUE constraint) also guarantees that no two rows have an the same value in the constrained column or combination of columns. However, a UNIQUE key column can contain null entries — typically one NULL is permitted because NULL is not considered equal to another NULL by the UNIQUE constraint. A table can have multiple UNIQUE keys.

In summary: a primary key is a unique key that additionally prohibits NULLs and is limited to one per table, while a unique key permits one NULL and a table can define many unique keys. Both create an underlying index that enforces uniqueness and speeds up lookups.


What is the difference between the DELETE TABLE and TRUNCATE TABLE commands in MySQL?

DELETE is a DML (Data Manipulation Language) command. It removes rows from a table based on an optional WHERE condition. With a WHERE clause it removes specific rows; without a WHERE clause it removes all rows. DELETE is transactional — it can be rolled back within an active transaction. DELETE fires any triggers defined on the table. It logs each deleted row individually, making it slower on large tables.

TRUNCATE is a DDL (Data Definition Language) command. It removes all rows from a table by deallocating the data pages in one operation. TRUNCATE cannot include a WHERE clause — it always removes each individual row. TRUNCATE is auto-committed and generally may not be rolled back. It skips row-level triggers. TRUNCATE is significantly faster than DELETE for large tables because it does not produce individual row-deletion log entries. TRUNCATE also resets the AUTO_INCREMENT counter to its initial value.

Main differences: DELETE can target specific rows with WHERE; TRUNCATE always removes all rows. DELETE can be rolled back; TRUNCATE is permanent. DELETE is slower on large tables; TRUNCATE is much faster. DELETE fires triggers; TRUNCATE does not.


What is meant by transaction? What are ACID properties?

A transaction is one logical unit of work that groups at least one SQL operation so that they all succeed or all fail together. A transaction is treated as indivisible — if any operation in it fails, the entire transaction is rolled back to the state before it began.

A classic example is a bank transfer: debiting one account and crediting another must both succeed. If the credit fails after the debit, the database would be in an incorrect state. A transaction prevents this by ensuring both operations happen together or neither does.

ACID properties are the four guarantees that every transaction must provide:

Atomicity means the transaction is treated as one unit. Either all statements in the transaction execute and are committed, or none of them take effect. Partial completion is not allowed.

Consistency means a transaction moves the database from one valid state to another valid state. All integrity constraints, rules, and cascades must be satisfied after the transaction. The database is never left in a logically inconsistent state.

Isolation means concurrent transactions do not interfere with each other. The intermediate state of one transaction is invisible to other transactions. Each transaction behaves as if it runs in isolation, even when hundreds of transactions execute at the same time.

Durability means that the same time a transaction is committed, its changes are permanent. The committed data is written to persistent storage and will survive system failures such as crashes, power cuts, or server restarts.


Which MySQL function is used to concatenate string?

The CONCAT() function serves to join two or more strings together into one string in MySQL.

Syntax: CONCAT(string1, string2, string3, ...)

Example: SELECT CONCAT('Hello', ' ', 'World'); returns Hello World. SELECT CONCAT(first_name, ' ', last_name) AS full_name FROM employees; joins the first and last name columns with a space between them.

If any argument passed to CONCAT() is NULL, the function returns NULL for the entire result. To handle NULLs gracefully, CONCAT_WS() (Concatenate With Separator) is used instead. It takes a separator as the first argument and ignores null entries among the remaining arguments.

Example with CONCAT_WS: SELECT CONCAT_WS(' ', first_name, last_name) AS full_name FROM employees; — even if one name part is NULL, the other part is still returned without the separator appearing next to nothing.


How can you change the name of any existing table by using the SQL statement?

To rename an existing table in MySQL, use the RENAME TABLE statement.

Syntax: RENAME TABLE old_table_name TO new_table_name;

Example: RENAME TABLE employees TO staff; This renames the employees table to staff. All data, indexes, constraints, and foreign key references are preserved under the new name.

You can rename multiple tables in one atomic statement: RENAME TABLE employees TO staff, departments TO divisions;

An alternative is the ALTER TABLE statement with RENAME TO: ALTER TABLE employees RENAME TO staff;

Both approaches produce the same result. RENAME TABLE is preferred when renaming multiple tables at the same time because the entire operation is atomic — either all tables are renamed successfully or none are renamed at all, preventing situations where only some renames complete.


What is the view? How can you create and drop view in MySQL?

A view is a virtual table in MySQL. It holds no actual data. Instead, it stores a SQL SELECT query. When you query a view, MySQL executes the underlying SELECT statement and presents the results as if they were coming from a real table. A view behaves like a table for all read operations.

Views are used to simplify complex queries by hiding joins and conditions behind a simple name, to restrict access to sensitive columns by exposing only certain columns through the view, and to reuse frequently needed query logic across multiple queries.

Creating a view: Syntax: CREATE VIEW view_name AS SELECT columns FROM table WHERE condition;

Example: CREATE VIEW high_salary_employees AS SELECT employee_id, first_name, salary FROM employees WHERE salary > 50000;

You then query it like a table: SELECT * FROM high_salary_employees;

To update or replace an existing view: CREATE OR REPLACE VIEW high_salary_employees AS SELECT ...;

Dropping a view: Syntax: DROP VIEW view_name; Example: DROP VIEW high_salary_employees;

Dropping a view removes only the view definition from the database. The underlying source tables and their data are not affected in any way.


What are the functions of commit and rollback statements?

COMMIT ends the current transaction and permanently saves all the changes made during that transaction to the database. Once a transaction is committed, the changes are written to disk and become visible to other transactions. A committed transaction may not be undone using ROLLBACK.

Example: START TRANSACTION; UPDATE accounts SET balance = balance - 1000 WHERE account_id = 1; UPDATE accounts SET balance = balance + 1000 WHERE account_id = 2; COMMIT; After COMMIT, both account updates are permanently saved.

ROLLBACK cancels all the changes made in the current transaction and restores the database to the state it was in before the transaction began. ROLLBACK is executed when an error occurs or when the business logic determines that the changes should not be saved.

Example: START TRANSACTION; UPDATE accounts SET balance = balance - 1000 WHERE account_id = 1; -- Error: account 2 does not exist ROLLBACK; After ROLLBACK, the debit to account 1 is reversed. The database is back to its state before the transaction.

Together, COMMIT and ROLLBACK are the core of transaction management and support the Atomicity principle of ACID: a transaction is either fully committed or fully rolled back — there is no partial state.


What's wrong with this query? SELECT department_id, count(*) FROM department WHERE count(*) > 5 GROUP BY department_id;

The query is incorrect because COUNT(*), an aggregate function, is used inside the WHERE clause. SQL prohibits aggregate functions in the WHERE clause.

The reason is the order of clause evaluation. WHERE is executed before GROUP BY. At the time WHERE runs, rows have not yet been grouped and aggregate functions have not yet been computed. The value of COUNT(*) is therefore unknown when WHERE is evaluated, which causes a syntax error.

The corrected query replaces WHERE with HAVING, which is evaluated after GROUP BY and after aggregation:

SELECT department_id, COUNT(*) FROM department GROUP BY department_id HAVING COUNT(*) > 5;

HAVING is the correct clause for filtering on aggregate results. This corrected query first groups all rows by department_id, counts the rows in each group, and then HAVING filters out any group with five or fewer rows.


Explain different types of Normalization.

Normalization is the process of organizing a database schema to eliminate data redundancy and ensure data integrity. It involves decomposing tables based on dependency rules.

Functional Dependency is the foundation of normalization. A column B is functionally dependent on column A if knowing A determines the value of B. As an illustration, knowing an employee_id determines the employee_name.

First Normal Form (1NF) requires that every column holds atomic (single, indivisible) values — no lists or sets in a cell. Each row must be unique. Each column must contain only one type of data. A table that stores multiple phone numbers in one cell like 9876543210, 9123456789 violates 1NF.

Second Normal Form (2NF) requires the table to already satisfy 1NF and additionally every non-key column must depend on the complete primary key, not just part of it. Partial dependencies must be removed. This is relevant only when the primary key is composite (made up of several column).

Third Normal Form (3NF) requires the table to satisfy 2NF and additionally there must be no transitive dependencies. A transitive dependency is when a non-key column depends on another non-key column instead of directly on the primary key. As an illustration, if a table has employee_id, department_id, and department_name, and department_name depends on department_id rather than on employee_id, that is transitive and should be moved to a separate departments table.

Boyce-Codd Normal Form (BCNF) is a stricter extension of 3NF. It requires that for every functional dependency X determines Y, X must be a superkey of the table. BCNF resolves certain anomalies involving overlapping candidate keys that 3NF leaves unresolved.


Mention the differences between UNION and UNION ALL.

UNION merges the outputs of two or more SELECT statements into one result set and automatically removes any duplicate rows. The output contains only unique rows.

UNION ALL merges the outputs of two or more SELECT statements and keeps each individual row including duplicates. No deduplication step is performed.

Both require the same number of columns in each SELECT and matching data types for corresponding columns.

UNION is slower than UNION ALL because it must sort and deduplicate the combined result set. UNION ALL is faster because it simply concatenates the results without any extra processing.

Example: SELECT city FROM customers UNION SELECT city FROM suppliers; Returns a list of all unique cities mentioned in either table.

SELECT city FROM customers UNION ALL SELECT city FROM suppliers; Returns every city from both tables including duplicates.

When to use which: use UNION when the two result sets may overlap and you want unique results. Use UNION ALL when you know there are no duplicates, or when you intentionally want all rows including duplicates, or when query performance is a priority.


Mention the difference between TRUNCATE and ROUND functions.

TRUNCATE(number, decimal_places) removes decimal digits beyond the specified position without any rounding. It always cuts toward zero regardless of the following digits. Example: TRUNCATE(3.7896, 2) returns 3.78 — the third and fourth decimals are simply dropped. Example: TRUNCATE(3.999, 0) returns 3, not 4 — no rounding occurs.

ROUND(number, decimal_places) rounds the number to the specified number of decimal places using standard mathematical rounding. If the digit immediately after the cutoff position is 5 or greater, the last kept digit is rounded up; if it is less than 5, the value is rounded down. Example: ROUND(3.7896, 2) returns 3.79 — the third decimal 9 rounds the second decimal up. Example: ROUND(3.999, 0) returns 4 — the first decimal 9 causes rounding up to 4.

The key distinction is that TRUNCATE always discards digits and never increases the value, while ROUND applies rounding rules and may increase the value. TRUNCATE(3.999, 0) gives 3 whereas ROUND(3.999, 0) gives 4.


Mention the difference between ISNULL() and IFNULL().

ISNULL(expression) checks whether the given expression is NULL. It accepts exactly one argument and gives 1 (true) if the expression is NULL, or 0 (false) if it is not NULL. ISNULL is a test — it tells you whether something is NULL but does not replace or change the value. Example: SELECT ISNULL(NULL); returns 1. Example: SELECT ISNULL(5); gives 0.

IFNULL(expression, replacement) is a substitution function that accepts two arguments. If the first argument is not NULL, it returns the first argument unchanged. If the first argument is NULL, it returns the second argument as a fallback value. Example: SELECT IFNULL(NULL, 'Not Available'); returns Not Available. Example: SELECT IFNULL(salary, 0) FROM employees; replaces any NULL salary with 0.

The primary difference is in purpose and number of arguments. ISNULL takes one argument and answers the question is this NULL with a 0 or 1. IFNULL takes two arguments and answers what value should be shown when this is NULL by substituting a meaningful default. ISNULL is used for checking; IFNULL is used for handling null entries in output.


What is referential integrity?

Referential integrity is a database property that ensures the relationships between tables remain valid and consistent at all times. It guarantees that a foreign key value in a dependent table always corresponds to an existing primary key value in the parent table, or is NULL.

Referential integrity prevents two categories of problems:

Orphan records occur when a row in the dependent table references a primary key value that no longer exists in the parent table. As an illustration, an order record containing a customer_id that was deleted from the customers table would be an orphan.

Dangling references occur when you try to delete or update a primary key in the parent table that child rows still reference.

In MySQL, referential integrity is enforced by defining FOREIGN KEY constraints. When a foreign key is declared, the database automatically validates every INSERT and UPDATE to the dependent table to confirm that the referenced value exists in the parent.

The ON DELETE and ON UPDATE options control what happens to child records when the parent changes: CASCADE automatically propagates the change — deleting a parent deletes all related child rows, updating a parent key updates all matching child foreign key values. SET NULL sets the foreign key column in the child to NULL when the parent row is deleted. RESTRICT prevents the parent from being deleted or updated if child rows still reference it. NO ACTION behaves like RESTRICT with deferred checking.


Mention the difference between WHERE and HAVING.

WHERE filters individual rows before any grouping takes place. It is evaluated early in the query pipeline, before GROUP BY. WHERE works on the raw, ungrouped rows of the table. You cannot use aggregate functions (COUNT, SUM, AVG, MIN, MAX) inside a WHERE clause because those values have not yet been calculated.

Example: SELECT * FROM employees WHERE salary > 50000; filters out individual employee rows where the salary is 50000 or below before any grouping.

HAVING filters groups of rows after GROUP BY has been applied and after aggregate functions have computed their results. HAVING is evaluated after aggregation. You can and should use aggregate functions inside HAVING.

Example: SELECT department_id, AVG(salary) AS avg_salary FROM employees WHERE salary > 20000 GROUP BY department_id HAVING AVG(salary) > 50000;

In this query, WHERE first removes employees earning 20000 or less from the individual rows. GROUP BY then groups the remaining employees by department. HAVING then removes any department where the average salary of the remaining employees is 50000 or below.

In summary: WHERE operates on rows before grouping; HAVING operates on groups after aggregation. WHERE comes before GROUP BY in the execution order; HAVING comes after.


Count total number of 'a' appearing in the mentioned phrase 'Great Learning'.

To count the total occurrences of the letter a in the phrase Great Learning in MySQL, the standard approach uses LENGTH and REPLACE together. The logic is: measure the original length of the string, remove all occurrences of the letter a from the string, measure the new length, and subtract the new length from the original. The difference is the number of times a appeared.

Query: SELECT LENGTH('Great Learning') - LENGTH(REPLACE(LOWER('Great Learning'), 'a', '')) AS count_of_a;

Step by step: LOWER('Great Learning') converts the string to great learning so the search is case-insensitive. REPLACE('great learning', 'a', '') removes every a, producing gret lerning. LENGTH('great learning') is 14 characters. LENGTH('gret lerning') is 12 characters. 14 minus 12 equals 2.

The letter a appears 2 times in Great Learning — once in Great and once in Learning.


After creating a table, how a unique constraint can be added to a column and how will you delete the same?

To add a UNIQUE constraint to an existing column after the table has already been created, use the ALTER TABLE statement with ADD CONSTRAINT.

Syntax to add: ALTER TABLE table_name ADD CONSTRAINT constraint_name UNIQUE (column_name);

Example: ALTER TABLE employees ADD CONSTRAINT uq_email UNIQUE (email); This adds a unique constraint named uq_email on the email column. From this point no two rows can share the same email value.

You can also add without specifying a name and let MySQL generate one: ALTER TABLE employees ADD UNIQUE (email);

To drop (remove) a UNIQUE constraint, use ALTER TABLE with DROP INDEX. In MySQL, unique constraints are implemented as unique indexes, so they are dropped using the index name.

Syntax to drop: ALTER TABLE table_name DROP INDEX constraint_name;

Example: ALTER TABLE employees DROP INDEX uq_email;

If no constraint name was specified when adding the unique key, MySQL typically uses the column name as the default index name: ALTER TABLE employees DROP INDEX email;

To see all indexes and constraint names on a table: SHOW INDEX FROM employees;


How correlated sub-query can be applied in Having clause? Explain with an example.

A correlated subquery in the HAVING clause is one where the inner query references a value from the parent query's current group. HAVING evaluates the subquery after GROUP BY has created the groups, so the correlated reference is to aggregate or group-level values.

Example: Find all departments where the average salary of employees in that department is higher than the overall company average salary.

SELECT department_id, AVG(salary) AS avg_dept_salary FROM employees GROUP BY department_id HAVING AVG(salary) > (SELECT AVG(salary) FROM employees);

Here the inner subquery SELECT AVG(salary) FROM employees is not correlated to the outer group (it returns the global average), but it sits inside HAVING filtering groups.

A fully correlated example: Find departments where the count of employees exceeds the average count per department.

SELECT department_id, COUNT(*) AS emp_count FROM employees GROUP BY department_id HAVING COUNT(*) > ( SELECT AVG(dept_count) FROM (SELECT COUNT(*) AS dept_count FROM employees GROUP BY department_id) AS dept_sizes );

For each group (department), HAVING evaluates the inner query using that group's aggregated context, keeping only departments whose employee count exceeds the average count across all departments. The subquery is re-evaluated for each group, which is the defining characteristic of a correlated usage.


Explain Unique Key with example and describe when to declare an attribute as UNIQUE key instead of PRIMARY KEY.

A UNIQUE key is a constraint that ensures every value in the constrained column is distinct — no two rows can have an the same value. A column with a UNIQUE constraint can still hold null entries (typically one NULL is allowed per unique column).

Example: In a customers table, declare the email column as UNIQUE because no two customers should have the same email address. CREATE TABLE customers ( customer_id INT PRIMARY KEY, email VARCHAR(100) UNIQUE, name VARCHAR(50) );

When to declare UNIQUE instead of PRIMARY KEY:

Use UNIQUE when the column must be distinct but is not the primary row identifier. email, phone_number, and national_id_number are examples where uniqueness is required but the table's official identifier is a surrogate key like customer_id.

Use UNIQUE when the column may occasionally hold NULL. PRIMARY KEY never allows NULL, but UNIQUE does. If a value is optional but must be unique whenever it is provided, UNIQUE is the right choice.

Use UNIQUE when there are multiple candidate keys. A table can have only one PRIMARY KEY but any number of UNIQUE constraints. All other candidate keys beyond the primary key should be enforced with UNIQUE.

Use PRIMARY KEY for the main row identifier used in foreign key references. Choosing PRIMARY KEY makes the referencing relationships explicit and ensures the referenced column always has a value.


Write a SQL query to find all duplicate emails in a table named Person.

To find duplicate emails in the Person table, group rows by the Email column and use HAVING to keep only those email values that appear more than once.

SELECT Email FROM Person GROUP BY Email HAVING COUNT(*) > 1;

Explanation: GROUP BY Email collects all rows sharing the same email into one group. COUNT(*) counts how many rows are in each group. HAVING COUNT(*) > 1 keeps only groups where the same email appears in two or more rows.

For the table containing (1, a@b.com), (2, c@d.com), (3, a@b.com), the query returns a@b.com because it is present in rows 1 and 3.

An alternative using a self-join: SELECT DISTINCT p1.Email FROM Person p1 JOIN Person p2 ON p1.Email = p2.Email AND p1.Id <> p2.Id;

This joins the table to itself to find pairs of different rows that share the same email.


How to add columns to a table in MySQL. Explain with proper example.

To add a new column to an existing table, use ALTER TABLE with the ADD COLUMN clause.

Syntax: ALTER TABLE table_name ADD COLUMN column_name datatype [constraints];

Example 1 — add a single column: ALTER TABLE employees ADD COLUMN phone_number VARCHAR(15); Adds a phone_number column of type VARCHAR(15) to the employees table. Existing rows get NULL in this new column.

Example 2 — add a column with a default and NOT NULL: ALTER TABLE employees ADD COLUMN status VARCHAR(10) NOT NULL DEFAULT 'Active'; The status column may not be NULL and defaults to Active for existing rows.

Example 3 — control the position of the new column: ALTER TABLE employees ADD COLUMN middle_name VARCHAR(30) AFTER first_name; The AFTER keyword places the new column immediately after first_name. ALTER TABLE employees ADD COLUMN employee_code INT FIRST; FIRST places the column as the very first column within the table.

Example 4 — add multiple columns in one statement: ALTER TABLE employees ADD COLUMN city VARCHAR(50), ADD COLUMN country VARCHAR(50);

Use DESCRIBE employees; after the ALTER to verify the new column was added correctly.


Given a scenario, an attribute named First Name consists of duplicate values and as well as NULL values. Is it possible to apply Group By clause on First Name?

Yes, GROUP BY can be applied to the First Name column even when it contains duplicate values and null entries, and it handles both correctly.

Duplicate values: GROUP BY is designed exactly for this. It collects all rows that share the same First Name value into one group. Having many rows with the same first name is not a problem — it is the whole point of grouping. The result will have one row per unique First Name value with aggregate functions applied to each group.

null entries: GROUP BY treats all missing entries as belonging to the same group. All rows where First Name is NULL are grouped together as one NULL group. This is a deliberate behavior in MySQL — even though NULL does not equal NULL in comparisons (NULL = NULL is UNKNOWN), GROUP BY treats NULLs as equal for grouping purposes.

Example: SELECT first_name, COUNT(*) AS count FROM employees GROUP BY first_name;

If the table has first_name values John, John, Alice, NULL, NULL, the result contains three groups: John with count 2, Alice with count 1, and NULL with count 2.

So yes, GROUP BY works correctly on a column with both duplicates and NULLs.


What is the difference between RANK() and DENSE_RANK()? Justify the answer with sample code.

Both RANK() and DENSE_RANK() are window functions that assign a numerical rank to each record based on a specified ordering. They differ in how they handle ties.

RANK() assigns the same rank to tied rows but then skips ranks for the number of tied rows, creating gaps in the sequence. If two rows tie for rank 1, the next row receives rank 3 — rank 2 is skipped.

DENSE_RANK() assigns the same rank to tied rows but does not skip any ranks. The next distinct value always receives the next consecutive integer. If two rows tie for rank 1, the next distinct value receives rank 2 — no gap.

Example: Salaries: 60000, 60000, 50000, 40000.

With RANK(): 60000 — rank 1 60000 — rank 1 50000 — rank 3 (rank 2 is skipped) 40000 — rank 4

With DENSE_RANK(): 60000 — rank 1 60000 — rank 1 50000 — rank 2 (consecutive, no gap) 40000 — rank 3

Sample SQL: SELECT employee_id, salary, RANK() OVER (ORDER BY salary DESC) AS rank_val, DENSE_RANK() OVER (ORDER BY salary DESC) AS dense_rank_val FROM employees;

Use DENSE_RANK when you need consecutive ranking for finding the second or Nth highest value. Use RANK when gaps in ranking are meaningful, such as competition standings where two tied first-place winners mean no second place exists.


How do you control the data in the child table when there is a change in the parent table? Explain with an example.

You control dependent table behavior using the ON DELETE and ON UPDATE options in the FOREIGN KEY constraint. These options specify what action the database should take on the child rows automatically when the referenced parent row is deleted or updated.

Available actions:

CASCADE: The change is automatically propagated to the dependent table. Deleting a parent row deletes all matching child rows. Updating the parent key updates all matching child foreign key values.

SET NULL: The foreign key column in the child is set to NULL when the parent row is deleted or updated. The child rows are not removed.

RESTRICT: The operation on the parent is blocked if any child rows reference it. An error is returned.

NO ACTION: Similar to RESTRICT — the delete or update is rejected if it would create orphan records.

Example: CREATE TABLE departments (dept_id INT PRIMARY KEY, dept_name VARCHAR(50));

CREATE TABLE employees ( emp_id INT PRIMARY KEY, name VARCHAR(50), dept_id INT, FOREIGN KEY (dept_id) REFERENCES departments(dept_id) ON DELETE CASCADE ON UPDATE CASCADE );

With this setup: if a department is deleted from the departments table, all employees in that department are automatically deleted from the employees table. If a dept_id is updated in departments, the corresponding dept_id values in all employee rows are automatically updated to match.


What is a foreign key? How to implement the same in MySQL?

A foreign key is a column (or set of columns) in a single table whose values reference the primary key of another table. It creates a logical link between the two tables and enforces referential integrity by ensuring every foreign key value either matches an existing primary key value in the parent table or is NULL.

The table containing the foreign key is called the dependent table and the table containing the referenced primary key is called the parent table.

Implementation in MySQL:

Method 1 — define the foreign key when creating the table: CREATE TABLE orders ( order_id INT PRIMARY KEY, customer_id INT NOT NULL, order_date DATE, FOREIGN KEY (customer_id) REFERENCES customers(customer_id) );

Method 2 — add a foreign key to an existing table using ALTER TABLE: ALTER TABLE orders ADD CONSTRAINT fk_orders_customer FOREIGN KEY (customer_id) REFERENCES customers(customer_id);

Method 3 — add with cascade options: ALTER TABLE orders ADD CONSTRAINT fk_orders_customer FOREIGN KEY (customer_id) REFERENCES customers(customer_id) ON DELETE CASCADE ON UPDATE CASCADE;

Prerequisites: the child and parent columns must have compatible data types. The parent column must have a PRIMARY KEY or UNIQUE constraint. For foreign keys to be enforced in MySQL, the tables must use the InnoDB storage engine.


Write a SQL query to get the second highest salary from the Employee table. If there is no second highest salary, then the query should return null.

Approach 1 — nested MAX subquery (most straightforward): SELECT MAX(Salary) AS SecondHighestSalary FROM Employee WHERE Salary < (SELECT MAX(Salary) FROM Employee);

The inner subquery finds the highest salary. The outer query finds the highest salary among all salaries strictly below that maximum. If only one distinct salary exists, the outer MAX returns NULL automatically.

Approach 2 — LIMIT with OFFSET: SELECT IFNULL( (SELECT DISTINCT Salary FROM Employee ORDER BY Salary DESC LIMIT 1 OFFSET 1), NULL ) AS SecondHighestSalary;

DISTINCT removes duplicate salary values. ORDER BY DESC sorts from highest to lowest. OFFSET 1 skips the first row (the highest) and LIMIT 1 takes the next one. IFNULL handles the case where no second-highest exists.

Approach 3 — DENSE_RANK window function: SELECT salary AS SecondHighestSalary FROM ( SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk FROM Employee ) ranked WHERE rnk = 2 LIMIT 1;

For the given table with salaries 100, 200, 300, all three approaches return 200.


If we drop a table, does it also drop related objects like constraints, indexes, columns, defaults, Views and Stored Procedures? State different types of constraints.

When you execute DROP TABLE, MySQL permanently removes the table and everything that is stored as part of the table itself. This includes all columns, all indexes (primary key, unique, and regular indexes), all constraints defined on the table (NOT NULL, DEFAULT, CHECK, PRIMARY KEY, UNIQUE, FOREIGN KEY constraints declared on the table), and all triggers on the table.

Objects outside the table are handled differently: Views that reference the dropped table are NOT automatically dropped. The view definition remains in the database, but querying it will produce an error because the source table no longer exists. Stored procedures and functions that reference the dropped table are NOT dropped. They remain but will fail when executed. Foreign key constraints in other tables that point to the dropped table will cause an error if FOREIGN_KEY_CHECKS is enabled. You must drop or alter the dependent tables first.

Types of constraints in SQL: NOT NULL ensures a column may not hold a missing entry — each individual row must supply a value. UNIQUE ensures all values in a column are distinct — no duplicates allowed, but one NULL is typically permitted. PRIMARY KEY uniquely identifies each record — combines NOT NULL and UNIQUE; only one primary key per table. FOREIGN KEY links a column to the primary key of another table and enforces referential integrity. CHECK validates column values against a Boolean condition — values violating the condition are rejected. DEFAULT sets an automatic value for a column when no value is explicitly provided during INSERT.


Explain multi-row operators for subqueries with an example.

Multi-row operators are used when a subquery returns several row. Standard single-value comparison operators like equals or greater-than cannot handle multi-row results. Multi-row operators are designed to evaluate a value against a set of values.

IN checks whether a value matches any value in the set returned by the inner query. Example: SELECT employee_id, name FROM employees WHERE department_id IN (SELECT department_id FROM departments WHERE location = 'Mumbai'); Returns all employees who belong to any department located in Mumbai.

NOT IN returns rows where the value does not match any value in the inner query result. Example: SELECT customer_id FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM orders); Returns customers who have never placed an order.

ANY (equivalently SOME) evaluates to TRUE if the comparison is true for at least one value in the inner query result. Example: SELECT salary FROM employees WHERE salary > ANY (SELECT salary FROM employees WHERE department_id = 5); Returns employees whose salary is greater than at least one salary in department 5.

ALL evaluates to TRUE only if the comparison is true for every value in the inner query result. Example: SELECT salary FROM employees WHERE salary > ALL (SELECT salary FROM employees WHERE department_id = 5); Returns employees whose salary is greater than every salary in department 5 — effectively employees earning more than the department 5 maximum.


Explain difference between WHERE Clause and GROUP BY Clause?

The WHERE clause and the GROUP BY clause serve entirely different purposes in SQL.

WHERE filters individual rows from the table based on a condition. It runs before any grouping takes place. WHERE works on every raw row of the table one at a time. Rows that do not satisfy the WHERE condition are excluded from all further processing. Aggregate functions may not be used in WHERE.

Example: SELECT * FROM employees WHERE salary > 50000; returns only individual employee rows where the salary exceeds 50000.

GROUP BY organizes the remaining rows (after WHERE) into groups based on matching values in at least a single columns. Each group collapses into one summary row when combined with aggregate functions like COUNT(), SUM(), AVG(). GROUP BY is executed after WHERE.

Example: SELECT department_id, COUNT(*) FROM employees GROUP BY department_id; groups employees by department and counts how many are in each department.

Order of execution: WHERE → GROUP BY → HAVING → SELECT → ORDER BY. This means WHERE filters individual rows first, then GROUP BY groups the filtered rows, then HAVING filters the resulting groups using aggregate conditions.

They can work together in one query: SELECT department_id, AVG(salary) FROM employees WHERE salary > 20000 GROUP BY department_id; first removes employees earning 20000 or less, then groups the remaining employees by department and calculates the average salary per group.


What is the result of the following command? DROP VIEW view_name. Is it possible to update the views? If yes, How, If not Why?

DROP VIEW view_name removes the view definition named view_name from the database. The view itself — which is just a stored SELECT query — is deleted from the database catalog. The underlying source tables and all data in them are completely unaffected.

Regarding updating views — it depends on the type of view:

Simple views can be updated. A simple view is created from a single table using a basic SELECT without GROUP BY, DISTINCT, aggregate functions, UNION, or subqueries. You can perform INSERT, UPDATE, and DELETE on a simple view and the changes are applied directly to the underlying source table.

Example of updating through a simple view: UPDATE employee_names_view SET first_name = 'Alice' WHERE employee_id = 101; This updates the first_name in the underlying employees table.

Complex views may not be updated. A view is not updatable if it uses: GROUP BY or HAVING. Aggregate functions such as SUM, COUNT, AVG. The DISTINCT keyword. UNION or UNION ALL. Subqueries in the SELECT list. Joins in most circumstances.

The reason complex views may not be updated is that MySQL is not always able to determine which source table rows to modify when the view is built from multiple tables or derived computations.

The WITH CHECK OPTION clause can be added to a simple view to make sure that any INSERT or UPDATE done through the view must still satisfy the view's WHERE condition. If an updated row would no longer appear in the view, the operation is rejected.


List the conditions when joins should be used instead of nested sub queries.

Joins are generally preferred over nested subqueries in the following situations:

When you need columns from multiple tables in the result set. A JOIN naturally combines columns from all joined tables in one output row. A subquery in the WHERE clause cannot return columns from the inner query table alongside the parent query columns.

When performance matters with large tables. The database query optimizer can use indexes across joined tables very effectively. Correlated subqueries can be slow because they re-execute once per outer row.

When joining more than two tables. Chaining three or more JOINs is cleaner and more readable than nesting multiple subqueries inside each other.

When using aggregate functions across multiple tables. A single query with GROUP BY and JOIN across tables is more efficient than computing aggregates in nested subqueries.

When the relationship is a simple foreign key match. Linking a dependent table to its parent through a JOIN on the foreign key is the natural and intended way to retrieve related data.

When the optimizer can eliminate joins or push predicates down. Explicit JOINs give the optimizer more information to work with than hidden subquery logic.

Subqueries remain appropriate when: the logic is a filter based on a calculated value (salary greater than the average), when using NOT IN or NOT EXISTS to find non-matching records, when the result needed is a scalar value for comparison, or when the inner query is logically independent of the parent query and the intent is clearer as a subquery.


What are the levels at which check constraints can be created? Justify your answer, on what all data types the constraints can be applied.

CHECK constraints can be created at two levels:

Column-level constraint is defined inline with the column definition. It applies only to the single column it is attached to and can reference only that column in its condition. Example: CREATE TABLE employees ( age INT CHECK (age >= 18), salary DECIMAL(10,2) CHECK (salary > 0) );

Table-level constraint is defined after all column definitions at the end of the CREATE TABLE block. It can reference multiple columns in the same table, making it suitable for cross-column validation. Example: CREATE TABLE bookings ( start_date DATE, end_date DATE, CONSTRAINT chk_dates CHECK (end_date > start_date) );

Both levels can also be added to existing tables: ALTER TABLE employees ADD CONSTRAINT chk_age CHECK (age >= 18 AND age <= 65);

Data types on which CHECK constraints can be applied:

Numeric types (INT, DECIMAL, FLOAT, DOUBLE) — useful for range checks such as price > 0 or quantity BETWEEN 0 AND 10000. Character types (VARCHAR, CHAR) — useful for value list checks such as status IN ('Active', 'Inactive') or format constraints. Date and Time types (DATE, DATETIME, TIMESTAMP) — useful for date range checks such as hire_date >= '2000-01-01' or ensuring end_date is after start_date. Boolean equivalents (TINYINT used as 0 or 1) — CHECK (active IN (0, 1)).

CHECK constraints can be applied to any data type that supports Boolean comparison expressions. From MySQL 8.0.16 onwards they are fully enforced.


How to add foreign keys in MySQL?

A foreign key can be added in MySQL in two ways:

Method 1 — during table creation: CREATE TABLE orders ( order_id INT PRIMARY KEY, customer_id INT NOT NULL, FOREIGN KEY (customer_id) REFERENCES customers(customer_id) ); The FOREIGN KEY clause names the child column and REFERENCES specifies the parent table and column.

Method 2 — to an existing table using ALTER TABLE: ALTER TABLE orders ADD CONSTRAINT fk_orders_customer FOREIGN KEY (customer_id) REFERENCES customers(customer_id); The CONSTRAINT keyword names the foreign key so it can be referenced later if you need to drop it.

Adding with CASCADE options: ALTER TABLE orders ADD CONSTRAINT fk_orders_customer FOREIGN KEY (customer_id) REFERENCES customers(customer_id) ON DELETE CASCADE ON UPDATE CASCADE; ON DELETE CASCADE automatically deletes child rows when the parent is deleted. ON UPDATE CASCADE automatically updates child foreign key values when the parent key changes.

Prerequisites: the child and parent columns must have compatible data types. The parent column must be a PRIMARY KEY or have a UNIQUE constraint. Both tables must use the InnoDB storage engine for foreign keys to be enforced.


Is it possible to have two primary keys in a table? Explain what is meant by alternate keys in RDBMS with an example.

No. A table can have only one primary key. This is a fundamental rule of relational database design. However, a primary key can be a composite key — made up of two or more columns together — but it still counts as one primary key.

Alternate Key: An alternate key is a candidate key that was not selected as the primary key.

To understand this, a candidate key is any column or combination of columns that can uniquely identify each individual row within the table and may not be NULL. A table can have multiple candidate keys.

When the database designer selects one candidate key to serve as the primary key, all the remaining candidate keys that were not chosen become alternate keys.

Example: Consider a students table with columns student_id, email, and phone_number, all of which uniquely identify a student.

All three are candidate keys: student_id, email, and phone_number. If student_id is chosen as the primary key, then email and phone_number become alternate keys.

Alternate keys are enforced using UNIQUE constraints: CREATE TABLE students ( student_id INT PRIMARY KEY, email VARCHAR(100) UNIQUE, phone_number VARCHAR(15) UNIQUE, name VARCHAR(50) );

Here, email and phone_number are alternate keys. They remain unique identifiers of a student but are not the designated primary key.


What is a view? Mention differences between Simple and Complex view.

A view is a virtual table whose content is defined by a stored SQL SELECT query. It holds no actual any data. When you query a view, MySQL executes the underlying SELECT and provides the results as if they came from a real table.

Views simplify complex queries, restrict access to sensitive columns, and allow the same query logic to be reused without rewriting it.

A Simple View is created from one source table using a basic SELECT statement. It contains no GROUP BY, no aggregate functions, no DISTINCT, no UNION, no subqueries, and typically no joins. Simple views are updatable — you can perform INSERT, UPDATE, and DELETE through a simple view and the changes are applied directly to the underlying base table.

Example of a simple view: CREATE VIEW employee_names AS SELECT employee_id, first_name, last_name FROM employees; You can update through this: UPDATE employee_names SET first_name = 'Ravi' WHERE employee_id = 10;

A Complex View is created using multiple tables, or involves GROUP BY, aggregate functions, DISTINCT, UNION, or subqueries. Complex views are generally not updatable because MySQL is not always able to determine how to map the change back to the correct rows in the source tables.

Example of a complex view: CREATE VIEW dept_headcount AS SELECT d.department_name, COUNT(e.employee_id) AS headcount FROM employees e JOIN departments d ON e.department_id = d.department_id GROUP BY d.department_name;

Main differences: simple views use a single table; complex views can use many. Simple views support DML; complex views generally do not. Simple views have no aggregation; complex views can. Simple views are easy to maintain; complex views encapsulate sophisticated business logic.


What is the difference between nested and correlated subqueries?

A nested subquery (non-correlated subquery) is an inner query that is completely independent of the parent query. It does not reference any column from the outer query. It can be executed automatically and produces a fixed result. The nested subquery runs once, its result is used by the outer query for all rows.

Example of a nested subquery: SELECT employee_id, salary FROM employees WHERE salary > (SELECT AVG(salary) FROM employees); The inner query SELECT AVG(salary) FROM employees runs once and gives back one value. The outer query uses that fixed value to filter all employees.

A correlated subquery references at least a single column from the parent query. Because it depends on the outer query's current row, it cannot run independently. It is re-executed once for each individual row processed by the outer query, each time using the current row's values.

Example of a correlated subquery: SELECT employee_id, salary, department_id FROM employees e WHERE salary > (SELECT AVG(salary) FROM employees WHERE department_id = e.department_id); For each employee in the parent query, the inner query runs with that employee's department_id, calculating the average salary for that specific department.

Main differences: A nested subquery runs once; a correlated subquery runs once per outer row. Nested subqueries are independent and can be tested alone; correlated subqueries depend on the parent query. Nested subqueries are generally faster; correlated subqueries can be slow on large tables because of repeated execution. Nested subqueries are simpler to read; correlated subqueries are more powerful for per-row comparisons.


Name the operator which is used in the query for pattern matching?

The LIKE operator is used in SQL for pattern matching. It is placed in the WHERE clause to search for a specified pattern in a character column.

LIKE uses two wildcard characters:

The percent sign % matches any sequence of zero or more characters. Example: SELECT * FROM customers WHERE name LIKE 'A%'; returns all customers whose name starts with the letter A. Example: SELECT * FROM products WHERE code LIKE '%SQL%'; returns all products whose code contains the string SQL anywhere.

The underscore _ matches exactly one character. Example: SELECT * FROM products WHERE code LIKE 'A_C'; matches any three-character code starting with A and ending with C, such as ABC or AXC.

NOT LIKE is the negation — it returns rows where the column does not match the pattern.

The ESCAPE clause is used when you need to search for a literal percent or underscore character: SELECT * FROM products WHERE description LIKE '50\%' ESCAPE ''; finds descriptions containing the literal text 50%.

For more powerful regular expression matching, MySQL also provides REGEXP (or RLIKE): SELECT * FROM employees WHERE name REGEXP '^[A-C]'; returns names starting with A, B, or C.


What are character manipulation functions? Give some examples.

Character manipulation functions are built-in SQL functions that operate on string (text) data. They transform, extract, search, or format character values.

UPPER(string) converts all characters to uppercase. Example: SELECT UPPER('hello'); returns HELLO.

LOWER(string) converts all characters to lowercase. Example: SELECT LOWER('WORLD'); returns world.

LENGTH(string) returns the number of characters in a string. Example: SELECT LENGTH('Database'); returns 8.

SUBSTRING(string, start, length) extracts a portion of a string starting at the given position for the specified number of characters. Example: SELECT SUBSTRING('Database', 1, 4); returns Data.

CONCAT(str1, str2, ...) joins two or more strings into one. Example: SELECT CONCAT('Data', 'base'); returns Database.

TRIM(string) removes leading and trailing spaces. Example: SELECT TRIM('  hello  '); returns hello.

REPLACE(string, from_str, to_str) replaces all occurrences of a substring with another string. Example: SELECT REPLACE('Hello World', 'World', 'SQL'); returns Hello SQL.

INSTR(string, substring) returns the position of the first occurrence of a substring within a string. Example: SELECT INSTR('Database', 'base'); returns 5.

LPAD(string, length, pad) and RPAD() pad a string to a specified length by prepending or appending a pad string. Example: SELECT LPAD('5', 4, '0'); gives 0005.


Explain Check Constraint in-detail with an example.

A CHECK constraint is a rule defined on a column or table that restricts which values can be stored. Any INSERT or UPDATE that would result in a value violating the CHECK condition is rejected by the database with a constraint error.

CHECK constraints ensure domain integrity by enforcing business rules at the database level, preventing invalid data regardless of where the data entry originates.

Column-level CHECK (applies to a single column): CREATE TABLE employees ( employee_id INT PRIMARY KEY, age INT CHECK (age >= 18 AND age <= 65), salary DECIMAL(10,2) CHECK (salary > 0) ); The age column only accepts values between 18 and 65. The salary column only accepts positive values.

Table-level CHECK (can reference multiple columns): CREATE TABLE products ( product_id INT PRIMARY KEY, price DECIMAL(10,2), discounted_price DECIMAL(10,2), CONSTRAINT chk_discount CHECK (discounted_price < price) ); This guarantees the discounted price is always lower than the original price.

Enum-style validation: CREATE TABLE orders ( order_id INT PRIMARY KEY, status VARCHAR(20) CHECK (status IN ('Pending', 'Shipped', 'Delivered', 'Cancelled')) ); Only the four listed values are accepted in the status column.

Adding a CHECK constraint to an existing table: ALTER TABLE employees ADD CONSTRAINT chk_age CHECK (age >= 18);

Important: MySQL fully enforces CHECK constraints from version 8.0.16 onwards. In earlier versions they were accepted syntactically but not actually enforced.


Compare sub-query and joins.

Subqueries and JOINs are both used to work with data from multiple tables, but they operate differently and are suited to different problems.

A subquery is a SELECT statement embedded inside another SQL statement. It is evaluated first and its result is used by the parent query. Subqueries can appear in the WHERE clause to filter rows, in the FROM clause as a derived table, or in the SELECT list as a scalar value. They are best used when the logic is a filter based on a calculated or looked-up value, or when checking for the presence or absence of related records with EXISTS or NOT EXISTS.

A JOIN combines columns from two or more tables into one result set based on a related column. JOINs return data from all joined tables in one row and are evaluated as part of the FROM clause. They are best used when you need columns from multiple tables in the output, when linking tables through foreign keys, or when aggregate operations span multiple tables.

Performance: JOINs are generally faster because query optimizers handle them well with indexed keys. Correlated subqueries can be slow as they re-execute for every outer row. Non-correlated subqueries execute once and are comparable to JOINs.

Readability: Subqueries can be easier to read for filtering logic like salary greater than average. JOINs are clearer when retrieving columns from multiple related tables.

Result shape: A JOIN expands the row width by adding columns from the joined table. A subquery in WHERE only filters rows — it does not add columns from the inner query.

Both can often produce identical results and the choice depends on readability, performance, and whether column output from the related table is needed.


What are the advantages of using Analytical functions?

Analytical functions (window functions) offer several important advantages over traditional aggregate functions and subquery-based approaches.

Row-level detail is preserved. Unlike GROUP BY which collapses multiple rows into one summary row, analytical functions return a value for each individual row while also computing group-level statistics. You can see both individual data and summary values in the same result without a second query.

Complex self-joins are eliminated. Before window functions, getting a running total or accessing a value from a previous row required joining a table to itself, which was verbose and hard to debug. Window functions achieve this in one, readable expression.

Cumulative and rolling calculations become simple. Running totals, cumulative revenue, and moving averages that previously required multiple subqueries can be expressed with one OVER clause and a ROWS BETWEEN frame specification.

Flexible partitioning without collapsing rows. PARTITION BY applies the function independently within each group (like GROUP BY), but every individual row still appears in the output.

Powerful ranking within groups. RANK(), DENSE_RANK(), and ROW_NUMBER() assign ranks within a partition without any complex self-join. Identifying the top three products per category is one query.

Time-series and sequential analysis is straightforward. LAG() and LEAD() directly access the previous or next row's value, enabling month-over-month comparisons and gap detection in one query.

Better performance in many cases. A single query with window functions often replaces several nested subqueries or self-joins, reducing the number of full table scans and improving execution time.


What is a view, and why use it? Can we create a view based on another view?

A view is a virtual table defined by a stored SQL SELECT query. It holds no actual data. When you query a view, MySQL executes its underlying SELECT at that moment and provides the result as if it came from a real table.

Why use views:

Simplification: complex multi-table joins and calculations can be hidden behind a simple view name. Users query the view with a basic SELECT without needing to understand the underlying complexity.

Security: views can expose only selected columns while keeping sensitive columns hidden. Access can be granted to a view without granting access to the source tables.

Reusability: common query patterns are defined once and reused across many queries and applications without repeating the logic.

Abstraction: if source table structures change, only the view definition needs updating, not every query that uses the view.

Consistency: a view guarantees that every user querying it gets results from the same verified logic.

Can a view be created based on another view? Yes. A view can reference another view instead of a source table. This is called a nested view or chained view.

Example: CREATE VIEW high_earners AS SELECT employee_id, salary FROM employees WHERE salary > 50000;

CREATE VIEW very_high_earners AS SELECT employee_id, salary FROM high_earners WHERE salary > 100000;

The second view references the first. When very_high_earners is queried, MySQL resolves the full chain of views back to the source table and executes the combined logic.

Nested views work but should be used carefully, as deep chains can make debugging, performance tuning, and understanding the data lineage more difficult.


What is normalization in SQL, and why use it?

Normalization is the systematic process of structuring a relational database schema to reduce data redundancy and ensure data integrity. It works by decomposing tables into smaller, well-focused tables and defining clear relationships among them based on dependency rules.

Why use normalization:

Eliminates data redundancy. Without normalization, the same piece of information might be stored in many rows. If a department name changes, it would need to be updated in every employee row. Normalization stores the department name once in a departments table and references it via a foreign key.

Prevents update anomalies. An update anomaly occurs when changing one occurrence of duplicated data but missing others, leaving the database in an inconsistent state.

Prevents insert anomalies. An insert anomaly occurs when you cannot add new data without also inserting unrelated or incomplete data.

Prevents delete anomalies. A delete anomaly occurs when removing a row accidentally destroys important related information that was stored in the same row.

Improves data integrity. Clear dependencies between tables and enforced foreign key relationships make sure that data remains accurate and consistent.

Makes the schema easier to understand and extend. Smaller, focused tables are easier to read, modify, and scale.

Normalization levels: 1NF — atomic values, no repeating groups, unique rows. 2NF — removes partial dependencies on composite primary keys. 3NF — removes transitive dependencies (non-key columns depending on other non-key columns). BCNF — a stricter version of 3NF, requiring every determinant to be a superkey.


What is the difference between INNER JOIN and LEFT JOIN in SQL? Provide an example of each.

An INNER JOIN returns only the rows where there is a matching record in both tables. Rows from either table that have no match in the other table are excluded entirely from the result.

Example of INNER JOIN: SELECT customers.customer_id, customers.name, orders.order_id FROM customers INNER JOIN orders ON customers.customer_id = orders.customer_id; This returns only customers who have at least one order. Customers with no orders and orders with no matching customer are both excluded.

A LEFT JOIN (LEFT OUTER JOIN) outputs all rows from the left (first) table and the matching rows from the right table. Where no match exists in the right table, the columns from the right table appear as NULL in the result. The left table rows are never excluded.

Example of LEFT JOIN: SELECT customers.customer_id, customers.name, orders.order_id FROM customers LEFT JOIN orders ON customers.customer_id = orders.customer_id; This returns every customer including those who have never placed an order. For customers with no orders, the order_id column shows NULL.

The key practical difference: INNER JOIN keeps only matched rows from both sides. LEFT JOIN always keeps all rows from the left table and fills in NULL where the right side has no match. LEFT JOIN is especially useful for finding records in a single table that have no corresponding records in another, by filtering for WHERE right_table.id IS NULL after the join.


Explain the concept of normalization in database design. Why is it important?

Normalization is the process of organizing a relational database to minimize redundancy and ensure logical data storage. It is achieved by applying a series of rules called normal forms, each building on the previous.

Why it is important:

Reduces data redundancy. The same fact is stored only once. When customer information is stored in a dedicated customers table rather than repeated across every order row, one update to the customer changes the data everywhere instantly.

Prevents data anomalies: Update anomaly: when the same data is stored in multiple places, updating one copy while missing another creates inconsistency. Insert anomaly: being unable to record one fact without also recording an unrelated fact. Delete anomaly: accidentally losing important data when an unrelated row is deleted.

Improves data integrity. Normalized schemas use foreign keys and clear dependencies to enforce that data is always valid and consistent.

Simplifies maintenance. Smaller, single-purpose tables are easier to understand, modify, query, and extend as the application grows.

Supports better query design. Well-normalized schemas join cleanly along defined relationships rather than requiring complex WHERE conditions to resolve stored redundancies.

Normal forms and what they address: First Normal Form (1NF) eliminates repeating groups and non-atomic values. Second Normal Form (2NF) removes partial dependencies from composite primary keys. Third Normal Form (3NF) removes transitive dependencies between non-key columns. Boyce-Codd Normal Form (BCNF) resolves complex dependency cases left by 3NF.


Define what a primary key is in a database. How does it differ from a foreign key?

A primary key is a column or combination of columns in a table that uniquely identifies each record. Every value in the primary key must be unique across all rows and may not be NULL. A table can have only one primary key. The primary key serves as the authoritative row identifier and is the main target for foreign key references from other tables.

Example: In an employees table, employee_id INT PRIMARY KEY uniquely identifies each employee.

A foreign key is a column in a single table whose values reference the primary key of another table. It establishes a relationship between the two tables and enforces referential integrity by ensuring every foreign key value either matches an existing primary key value in the parent table or is NULL.

Example: In an orders table, customer_id is a foreign key referencing customer_id in the customers table.

Differences:

Purpose: the primary key identifies rows within its own table. The foreign key creates a relationship between two tables by pointing to the primary key of the parent table.

Uniqueness: primary key values must be unique per table. Foreign key values can repeat — many rows in the dependent table can reference the same parent row (for example, many orders can belong to the same customer).

null entries: the primary key never allows NULL. The foreign key can be NULL, meaning the row is not linked to any parent.

Count: one primary key per table. A table can have multiple foreign keys, each pointing to a different parent table.

Location: the primary key resides in the parent table. The foreign key resides in the dependent table and points to the parent.


What are SQL aggregate functions? List at least three aggregate functions and describe their purposes.

Aggregate functions are built-in SQL functions that perform a calculation on a set of rows and return one summary value. They operate on groups of rows, ignoring null entries in most cases, and are commonly used with GROUP BY.

COUNT() counts the number of rows or non-null entries. COUNT(*) counts all rows including those with missing entries. COUNT(column) counts only rows where the specified column is not NULL. Example: SELECT COUNT(*) FROM employees; returns the total number of employees. Example: SELECT department_id, COUNT(*) FROM employees GROUP BY department_id; counts employees per department.

SUM() returns the total of all non-NULL numeric values in a column. Example: SELECT SUM(salary) FROM employees; returns the total payroll. Example: SELECT department_id, SUM(salary) FROM employees GROUP BY department_id; returns total salary per department.

AVG() returns the arithmetic mean of all non-NULL numeric values in a column. Example: SELECT AVG(salary) FROM employees; returns the average salary across all employees.

MIN() returns the smallest non-missing entry in a column. Works on numbers, text (alphabetically), and dates. Example: SELECT MIN(hire_date) FROM employees; returns the earliest hiring date.

MAX() returns the largest non-missing entry in a column. Example: SELECT MAX(salary) FROM employees; returns the highest salary.

All aggregate functions except COUNT(*) ignore null entries in their computation.


How does the GROUP BY clause work in SQL? In which situations would you use it?

The GROUP BY clause organizes rows that share an the same value in at least one specified columns into groups. Once rows are grouped, aggregate functions like COUNT(), SUM(), AVG(), MIN(), MAX() are applied to each group separately, and one summary row is produced per group.

How it works: SQL first filters rows with WHERE, then GROUP BY collects rows sharing an the same value in the grouping columns, then aggregate functions compute results within each group, then HAVING optionally filters those groups, and finally the result is returned.

Syntax: SELECT column, aggregate_function(column) FROM table WHERE condition GROUP BY column HAVING aggregate_condition;

Example: Count employees per department: SELECT department_id, COUNT(*) AS employee_count FROM employees GROUP BY department_id; Each unique department_id becomes one row in the output, with a count of employees in that department.

Important rule: every column in the SELECT list that is not wrapped in an aggregate function must appear in the GROUP BY clause.

Situations to use GROUP BY:

When generating per-category summaries: total sales per region, average salary per department, count of orders per customer.

When finding groups that meet a threshold using HAVING: departments with more than 10 employees, products with total sales above 100000.

When creating reports that require subtotals: monthly revenue breakdown, product-line profitability by year.

When ranking or comparing groups: which department has the highest average salary, which month had the most orders.


Describe the purpose of the ORDER BY clause in an SQL query. How is it different when sorting in ascending versus descending order?

The ORDER BY clause sorts the rows in the result set of a SELECT query by at least one specified columns. Without ORDER BY, SQL does not guarantee any particular order for the rows returned — the database can return them in any order.

Ascending order (ASC) sorts rows from the smallest value to the largest. For numbers, this is lowest to highest. For text, this is alphabetical from A to Z. For dates, this is earliest date first. ASC is the default and requires no to be written explicitly. Example: SELECT * FROM employees ORDER BY salary; or SELECT * FROM employees ORDER BY salary ASC; — both list employees from lowest to highest salary.

Descending order (DESC) sorts rows from the largest value to the smallest. For numbers, highest to lowest. For text, reverse alphabetical Z to A. For dates, most recent date first. DESC must be written explicitly. Example: SELECT * FROM employees ORDER BY salary DESC; — lists the highest earners first.

You can sort by multiple columns: SELECT * FROM employees ORDER BY department_id ASC, salary DESC; This sorts first by department in alphabetical order, and within each department sorts by salary from highest to lowest.

NULL handling: in MySQL, null entries are treated as lower than any non-missing entry in ascending order (they appear first) and as higher in descending order (they appear last).

ORDER BY is the final clause evaluated in a SELECT query — it runs after WHERE, GROUP BY, HAVING, and SELECT.


What is a subquery in SQL? Provide an example situation where a subquery might be useful.

A subquery is a SELECT statement nested inside another SQL statement, called the parent query. The subquery executes first and its result is passed to the outer query. Subqueries can appear in the WHERE clause, FROM clause, SELECT list, or HAVING clause.

Types of subqueries: Single-row subquery outputs one row and a single column, used with = or comparison operators. Multi-row subquery returns multiple rows, used with IN, ANY, ALL, or EXISTS. Correlated subquery references a column from the parent query and re-executes for each outer row. Scalar subquery gives back a single value and can appear in the SELECT list.

Example situation 1 — Filtering based on an aggregate value: Find all employees earning more than the company average salary. SELECT employee_id, first_name, salary FROM employees WHERE salary > (SELECT AVG(salary) FROM employees); The subquery computes the average once; the parent query uses it to filter employees. This cannot easily be done without a subquery.

Example situation 2 — Finding non-matching records: Find customers who have never placed an order. SELECT customer_id, name FROM customers WHERE customer_id NOT IN (SELECT DISTINCT customer_id FROM orders);

Example situation 3 — Row-by-row comparison using a correlated subquery: Find employees who earn more than the average salary in their own department. SELECT employee_id, salary FROM employees e WHERE salary > (SELECT AVG(salary) FROM employees WHERE department_id = e.department_id);

Each of these situations involves a value that must first be calculated from another table or from the same table before it can be used for filtering.


Describe the role of the HAVING clause in SQL. How is it different from the WHERE clause?

The HAVING clause filters groups of rows that result from a GROUP BY operation. It applies conditions to aggregated results after grouping has been performed. HAVING allows questions such as: show only departments with more than 5 employees, or show only product categories with an average price above 100.

Syntax: SELECT column, aggregate_function(column) FROM table GROUP BY column HAVING aggregate_condition;

Example: SELECT department_id, COUNT(*) AS emp_count FROM employees GROUP BY department_id HAVING COUNT(*) > 5; This groups employees by department, counts them per group, and returns only departments with more than 5 employees.

Difference from WHERE:

WHERE filters individual rows before grouping. It is evaluated before GROUP BY. WHERE may not contain aggregate functions because aggregation has not yet happened. WHERE removes specific rows from the table before any groups are formed.

HAVING filters groups after GROUP BY and after aggregate functions have been computed. HAVING can and is intended to contain aggregate functions. HAVING removes groups whose aggregate condition fails.

They can work together: SELECT department_id, AVG(salary) FROM employees WHERE salary > 20000 GROUP BY department_id HAVING AVG(salary) > 50000; WHERE first removes employees earning 20000 or less, GROUP BY then groups the remaining rows by department, and HAVING then removes departments where the average of the remaining salaries is 50000 or below.

HAVING without GROUP BY is permitted and treats the entire table as one group.


What is the purpose of the DISTINCT keyword in SQL? How does it affect the results of a query?

The DISTINCT keyword eliminates duplicate rows from the result set of a SELECT query. When included, only unique combinations of the selected columns are returned — if two or more rows would produce the same values in all selected columns, only one of them appears in the output.

Syntax: SELECT DISTINCT column1, column2 FROM table;

Effect on results: Without DISTINCT: SELECT department_id FROM employees; outputs one row per employee. If 50 employees are spread across 5 departments, 50 rows are returned with many repeated department IDs. With DISTINCT: SELECT DISTINCT department_id FROM employees; returns 5 rows — one for each unique department ID.

When multiple columns are selected, DISTINCT applies to the combination of all selected columns together, not to each column independently. Example: SELECT DISTINCT city, country FROM customers; This returns each unique city-country pair, not just unique cities.

DISTINCT with COUNT: SELECT COUNT(DISTINCT department_id) FROM employees; counts how many unique departments exist, not how many employees.

Common use cases: Listing all unique values in a column such as all distinct countries or product categories. Counting unique values with COUNT(DISTINCT column). Removing accidental or intentional duplicate rows from a result set. Displaying available options without repetition in dropdown lists or reports.

DISTINCT adds a sort or deduplication step internally, which can slow down queries on large result sets.


Explain the significance of ACID properties in SQL.

ACID properties are the four fundamental guarantees that define what a reliable database transaction must provide. They ensure correctness, consistency, and fault tolerance in multi-user database systems.

Atomicity: A transaction is treated as one indivisible unit of work. Either every operation in the transaction is executed and committed, or none of them take effect. If a transaction fails midway, all completed operations within it are automatically rolled back. This prevents partial updates that would leave the database in a corrupted state. As an illustration, if a bank transfer debits one account but fails before crediting another, Atomicity ensures the debit is also reversed.

Consistency: A transaction always moves the database from one valid state to another valid state. Every integrity constraint, rule, trigger, and cascade must be satisfied both before and after the transaction completes. The database can never be left in a logically inconsistent or rule-violating state by any transaction.

Isolation: Concurrent transactions execute as if they were running one at a time. The intermediate state of one transaction is not visible to other transactions until it is committed. This prevents problems like dirty reads (reading uncommitted changes), non-repeatable reads, and phantom reads. Isolation levels such as READ COMMITTED, REPEATABLE READ, and SERIALIZABLE control the degree of isolation applied.

Durability: Once a transaction is committed, its changes are permanent. The data is written to persistent storage and will survive system failures, crashes, or power losses. A committed transaction may not be undone by any subsequent failure.

Together, ACID properties guarantee that the database behaves reliably and predictably even when errors occur or many transactions run at the same time.


What is a foreign key constraint, and why is it used?

A foreign key constraint is a rule that links a column (or set of columns) in a dependent table to the primary key of a parent table. It ensures referential integrity by guaranteeing that every value in the foreign key column either matches an existing primary key value in the parent table or is NULL.

Example: CREATE TABLE orders ( order_id INT PRIMARY KEY, customer_id INT, FOREIGN KEY (customer_id) REFERENCES customers(customer_id) ); The customer_id column in orders is a foreign key. Every order must belong to a customer that exists in the customers table.

Why foreign key constraints are used:

Enforcing referential integrity: they prevent orphan records. You cannot insert an order with a customer_id that does not exist in the customers table, and you cannot delete a customer who still has orders (unless CASCADE is configured).

Cascading operations: with ON DELETE CASCADE or ON UPDATE CASCADE, the database automatically propagates changes from the parent to child rows. Deleting a customer can automatically delete all their orders, keeping the database consistent without manual intervention.

Documenting relationships: foreign keys make the logical relationships between tables explicit in the schema, improving clarity and making JOIN conditions obvious.

Preventing data inconsistency: without foreign keys, application bugs could insert invalid references that corrupt the logical structure of the data without the database detecting it.

Improving query reliability: foreign keys indicate exactly how tables should be joined, reducing the risk of incorrect join conditions in queries.


Describe 1NF, 2NF, and 3NF in database normalization.

First Normal Form (1NF) requires that a table satisfies these conditions: every column holds only atomic (single, indivisible) values with no lists or sets stored in one cell. Each column contains only one value per row. Each column has a unique name. All values in a column are of the same data type. Every row is unique (there is a primary key). A table that stores multiple phone numbers in one cell such as 9876543210, 9123456789 violates 1NF because the value is not atomic. The fix is to create a separate row for each phone number or move them to a related table.

Second Normal Form (2NF) requires the table to already be in 1NF and additionally every non-key column must be fully functionally dependent on the entire primary key — no partial dependencies. Partial dependency means a non-key column depends on only part of a composite (multi-column) primary key rather than on the whole key. This is relevant only when the primary key consists of two or more columns. Example: if a table has a composite primary key of (student_id, course_id) and a column student_name, student_name depends only on student_id not on the full key. This partial dependency must be resolved by moving student_name to a separate students table.

Third Normal Form (3NF) requires the table to already be in 2NF and additionally there must be no transitive dependencies. A transitive dependency exists when a non-key column A depends on another non-key column B, which itself depends on the primary key. Example: in a table with columns employee_id (primary key), department_id, and department_name, department_name depends on department_id and department_id depends on employee_id. Department_name is transitively dependent on the primary key through department_id. The fix is to move department_id and department_name to a separate departments table and reference it via a foreign key.


What is a correlated subquery?

A correlated subquery is a subquery that references at least a single columns from the outer (enclosing) query. Because it uses a value from the parent query's current row, the inner query may not be executed independently independently — it must be re-evaluated once for each individual row the outer query processes.

However, a regular (non-correlated) subquery is fully independent of the parent query and runs only once.

Example: SELECT employee_id, first_name, salary FROM employees e_outer WHERE salary > ( SELECT AVG(salary) FROM employees WHERE department_id = e_outer.department_id );

In this query, the inner subquery references e_outer.department_id from the parent query. For each employee row processed by the outer query, the inner query runs with that employee's department_id and calculates the average salary for that specific department. The outer query then checks whether the current employee's salary exceeds their department's average.

Correlated subqueries are used with EXISTS and NOT EXISTS to check whether related records exist, with comparison operators for per-row calculations, and inside HAVING for group-level correlated filtering.

The main trade-off is performance: because the inner query runs once per outer row, correlated subqueries can be slow on large tables. They can often be rewritten as JOINs with GROUP BY for better performance, but correlated subqueries are sometimes clearer to read and write.


Explain the WITH clause and provide an example.

The WITH clause defines a Common Table Expression (CTE), which is a temporary named result set that exists only for the duration of the query. The CTE is defined at the beginning of the query and can be referenced at least one times within the main SELECT, UPDATE, INSERT, or DELETE that follows.

Syntax: WITH cte_name AS ( SELECT columns FROM table WHERE condition ) SELECT * FROM cte_name WHERE further_condition;

Benefits of the WITH clause: Improves readability by breaking a complex query into named, understandable steps. Avoids repeating the same subquery multiple times in one statement. Allows multiple CTEs to be chained, with each CTE able to reference previously defined CTEs. Supports recursive queries for hierarchical data (WITH RECURSIVE).

Example 1 — Finding employees with above-average salary: WITH company_avg AS ( SELECT AVG(salary) AS avg_sal FROM employees ) SELECT employee_id, first_name, salary FROM employees CROSS JOIN company_avg WHERE salary > avg_sal;

The CTE company_avg computes the average salary once and gives it a name that can be used cleanly in the WHERE clause.

Example 2 — Chaining two CTEs: WITH dept_totals AS ( SELECT department_id, SUM(salary) AS total_salary FROM employees GROUP BY department_id ), top_departments AS ( SELECT department_id FROM dept_totals WHERE total_salary > 500000 ) SELECT * FROM employees WHERE department_id IN (SELECT department_id FROM top_departments);

The first CTE computes department salary totals. The second CTE filters to high-spending departments. The main query retrieves employees from those departments.


What are views in MySQL and why are they used?

A view in MySQL is a virtual table defined by a stored SQL SELECT query. It holds no actual any data. When you query a view, MySQL executes the underlying SELECT statement at that moment and provides the results as if they came from a regular table.

Why views are used:

Simplification: complex multi-table joins, subqueries, and business logic can be encapsulated in a view. Users can query the view with a simple SELECT * without needing to know the underlying complexity.

Security and access control: a view can expose only selected columns of a table, hiding sensitive data like passwords, salaries, or personal identifiers. Database administrators can grant users access to the view without granting access to the source tables.

Reusability: frequently used queries are written once as a view and referenced in multiple places, avoiding code duplication and ensuring consistent logic across applications.

Data abstraction: if the source table structure changes — columns renamed, tables split — only the view definition needs updating. Applications using the view continue to work without modification.

Consistency: every user who queries the same view gets results produced by the same verified query logic.

Creating a view: CREATE VIEW active_employees AS SELECT employee_id, first_name, department_id FROM employees WHERE status = 'Active';

Querying the view: SELECT * FROM active_employees;

Dropping a view: DROP VIEW active_employees; This removes only the view definition. The underlying employees table is unaffected.


State the difference between Primary key and candidate key.

A Candidate Key is any column or combination of columns that can uniquely identify each record within the table. A table can have multiple candidate keys. Each one qualifies to become the primary key.

A Primary Key is the candidate key officially chosen as the main row identifier. A table can have only one primary key. The primary key column prohibits null entries and does not allow duplicate values.

A table can have only one primary key but multiple candidate keys. The primary key may not hold NULL whereas a candidate key not chosen as primary may allow NULLs. All primary keys are candidate keys but not all candidate keys are primary keys. The primary key is automatically enforced by the RDBMS through a unique index and is used as the target for foreign key references from other tables.


What is a cross join?

A cross join produces the Cartesian product of two tables. It combines each individual row from the first table with every row from the second table. If the first table has M rows and the second has N rows, the result contains M multiplied by N rows.

A cross join requires no any join condition — there is no ON clause. Every possible combination of rows from both tables is returned.

Syntax: SELECT * FROM table1 CROSS JOIN table2;

Example: If an employees table has 10 rows and a departments table has 5 rows, the cross join produces 50 rows.

Cross joins are rarely used in production because they generate very large result sets. They are mainly used for generating all possible combinations, building test data, or constructing calendar or matrix structures.


What are LEAD and LAG functions? Explain.

LAG() and LEAD() are window functions that let you access values from other rows relative to the current row without needing a self-join.

LAG(column, offset, default) returns the value from a row that is a given number of positions before the current row in the ordered partition. The default offset is 1. If no prior row exists at that offset the default value is returned instead.

Example: SELECT employee_id, hire_date, LAG(hire_date, 1) OVER (ORDER BY hire_date) AS prev_hire_date FROM employees; — shows each employee's hire date alongside the hire date of the employee hired just before them.

LEAD(column, offset, default) is the opposite — it returns the value from a row that is a given number of positions after the current row.

Example: SELECT invoice_id, invoice_date, LEAD(invoice_date, 1) OVER (PARTITION BY customer_id ORDER BY invoice_date) AS next_invoice_date FROM invoices; — shows each invoice date alongside that customer's next invoice date.

Both functions are used with OVER() and ORDER BY. PARTITION BY can be added to reset the window per group. Common uses include month-over-month comparisons, calculating time gaps between consecutive events, and detecting changes from one row to the next.


Explain EXISTS operator with an example.

The EXISTS operator tests whether a subquery returns at least one row. It evaluates to TRUE if the inner query produces one or more rows and FALSE if the inner query gives back no rows. EXISTS is purely a membership check — it does not return values from the subquery.

Syntax: SELECT columns FROM table WHERE EXISTS (subquery);

Example: Find all customers who have placed at least one order. SELECT customer_id, customer_name FROM customers WHERE EXISTS (SELECT 1 FROM orders WHERE orders.customer_id = customers.customer_id);

The convention SELECT 1 inside EXISTS is used because EXISTS only cares whether any row is returned, not what value is returned. You could write SELECT *, SELECT id, or SELECT 1 and the result is identical.

NOT EXISTS is the negation — it evaluates to TRUE when the inner query gives back no rows. This serves to find records that have no match in another table. Example: SELECT customer_id FROM customers WHERE NOT EXISTS (SELECT 1 FROM orders WHERE orders.customer_id = customers.customer_id); — returns customers who have never placed an order.

EXISTS is typically faster than IN for large datasets because it short-circuits: it stops the moment the first matching row is found rather than evaluating the entire subquery result set.


What is the difference between isnull operator() and ifnull() function?

ISNULL(expression) takes exactly one argument and checks whether that expression is NULL. It gives 1 if the value is NULL or 0 if it is not NULL. ISNULL is a test — it does not replace the NULL with anything, it simply tells you whether the value is NULL.

Example: SELECT ISNULL(NULL); gives 1. SELECT ISNULL(5); gives 0.

IFNULL(expression, replacement) takes two arguments. If the first argument is not NULL it returns the first argument unchanged. If the first argument is NULL it returns the second argument as a substitute value.

Example: SELECT IFNULL(NULL, 'Not Available'); returns Not Available. SELECT IFNULL(salary, 0) FROM employees; replaces any NULL salary with 0 in the result.

The fundamental distinction is purpose and behaviour. ISNULL answers the question of whether a value is NULL by returning 0 or 1. IFNULL answers the question of what to display when a value is NULL by returning a fallback. ISNULL is used in conditional logic; IFNULL serves to substitute meaningful defaults for null entries in output.


What is the ACID property in a database?

ACID stands for Atomicity, Consistency, Isolation, and Durability. These four properties define the guarantees that every reliable database transaction must provide.

Atomicity means a transaction is all or nothing. Either every operation in the transaction executes and is committed successfully, or none of the operations are applied. If any step fails the entire transaction is automatically rolled back to the state before it began.

Consistency means a transaction always moves the database from one valid state to another valid state. All integrity constraints, rules, and cascades must be satisfied after the transaction completes. The database can never be left in a logically invalid or corrupted state.

Isolation means concurrent transactions execute independently of each other. The intermediate state of one transaction is not visible to other transactions. Each transaction behaves as though it is the only transaction running, even when hundreds of transactions execute at the same time.

Durability means that the same time a transaction is committed, its changes are permanent. The committed data is written to persistent storage and will survive system failures including crashes, power outages, and server restarts.


What is the difference between the RANK() and DENSE_RANK() functions?

Both RANK() and DENSE_RANK() are window functions that assign a numerical rank to rows based on a specified ordering. They differ only in how they handle tied values.

RANK() gives tied rows the same rank and then skips the subsequent rank numbers equal to the count of tied rows, creating gaps in the sequence. If two rows tie for rank 1, the next row receives rank 3 and rank 2 is skipped entirely.

DENSE_RANK() also gives tied rows the same rank but never skips any rank numbers. The next distinct value always receives the immediately following integer. If two rows tie for rank 1, the next distinct value receives rank 2 with no gap.

Example with salaries 60000, 60000, 50000, 40000: RANK():       60000 → 1, 60000 → 1, 50000 → 3, 40000 → 4 DENSE_RANK(): 60000 → 1, 60000 → 1, 50000 → 2, 40000 → 3

Sample SQL: SELECT employee_id, salary, RANK() OVER (ORDER BY salary DESC) AS rnk, DENSE_RANK() OVER (ORDER BY salary DESC) AS dense_rnk FROM employees;

Use DENSE_RANK when you need consecutive numbering with no gaps — for example, finding the second highest salary using WHERE dense_rnk = 2. Use RANK when gaps are meaningful, such as competition standings where two joint winners in first place means no second place exists.
'''